# Aula 22 — Rapidly-exploring Random Trees (RRT) — Planejamento de Trajetória do AGV na Fábrica de Paçoca

**Disciplina:** ECAA08 — Automática (2026.2) — UNIFEI  
**Projeto:** SCADA-Core Automática / Linha de Produção de Paçoca  
**Equipe:** Grupo 7  

Este notebook implementa e valida os algoritmos **RRT** e **RRT\*** para planejamento de trajetória do AGV de coleta de amostras no núcleo fabril da linha de paçoca, complementando o roteamento hamiltoniano (Aula 17) com a navegação livre de colisões no mapa contínuo 2D.

## Objetivos
1. Modelar o mapa da fábrica (obstáculos dos Setores 200–400) em $\mathcal{C} = [0,20] \times [0,12]\,\text{m}$.
2. Implementar o **RRT** com verificação de colisão, Voronoi Bias e reconstrução do caminho.
3. Implementar o **RRT\*** com escolha de pai ótimo e *rewiring*.
4. Verificar que a árvore $T$ satisfaz as propriedades formais (aciclia, conexidade, $|E|=|V|-1$, livre de colisão).
5. Comparar custo de caminho RRT vs. RRT\* em função de $N$ e simular replanejamento dinâmico.

---
## 1. Importações e Configuração

In [ ]:
import math
import random
import time
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Bibliotecas carregadas. Semente aleatória:", SEED)

---
## 2. Modelagem do Mapa Fabril e dos Obstáculos

O corredor do AGV é modelado como um retângulo de $20 \times 12\,\text{m}$.
Os obstáculos representam equipamentos fixos dos Setores 200 a 400.

| ID | Equipamento | Centro $(x_c, y_c)$ [m] | Largura × Altura [m] | Setor |
| :---: | :--- | :---: | :---: | :---: |
| O1 | Forno de Torra (CLP 02) | (3,5; 8,0) | 4,0 × 3,0 | S200 |
| O2 | Moinhos MOI-301A/B | (9,0; 8,5) | 3,5 × 2,5 | S300 |
| O3 | Homogeneizador HOM-301 | (13,0; 7,0) | 2,5 × 2,5 | S300 |
| O4 | Prensa PRN-401 | (17,0; 8,0) | 2,5 × 3,0 | S400 |
| O5-O8 | Colunas estruturais (círculos) | diversas | ⌀ 0,6 m | — |

In [ ]:
# ---------------------------------------------------------------------------
# Definição do mapa
# ---------------------------------------------------------------------------
MAP_W = 20.0   # largura [m]
MAP_H = 12.0   # altura [m]
R_AGV = 0.35   # raio do disco AGV [m]

@dataclass
class RectObs:
    """Obstáculo retangular: (cx, cy) = centro, w = largura, h = altura."""
    cx: float; cy: float; w: float; h: float; label: str

    def colide(self, x: float, y: float, r: float = R_AGV) -> bool:
        """Verifica se o disco de raio r centrado em (x, y) intercepta o retângulo."""
        # Ponto mais próximo do retângulo ao centro do disco
        px = max(self.cx - self.w / 2, min(x, self.cx + self.w / 2))
        py = max(self.cy - self.h / 2, min(y, self.cy + self.h / 2))
        return math.hypot(x - px, y - py) < r

@dataclass
class CircObs:
    """Obstáculo circular: (cx, cy) = centro, r_obs = raio do obstáculo."""
    cx: float; cy: float; r_obs: float; label: str

    def colide(self, x: float, y: float, r: float = R_AGV) -> bool:
        return math.hypot(x - self.cx, y - self.cy) < (self.r_obs + r)

# Obstáculos fixos do mapa
OBSTACULOS = [
    RectObs(3.5,  8.0,  4.0, 3.0, "Forno de Torra (S200)"),
    RectObs(9.0,  8.5,  3.5, 2.5, "Moinhos MOI-301A/B (S300)"),
    RectObs(13.0, 7.0,  2.5, 2.5, "Homogeneizador HOM-301 (S300)"),
    RectObs(17.0, 8.0,  2.5, 3.0, "Prensa PRN-401 (S400)"),
    # Colunas estruturais
    CircObs( 5.0, 2.0, 0.30, "Coluna C1"),
    CircObs( 8.0, 2.0, 0.30, "Coluna C2"),
    CircObs(11.0, 2.0, 0.30, "Coluna C3"),
    CircObs(14.0, 2.0, 0.30, "Coluna C4"),
]

# Pontos de interesse do AGV (ISA-5.1)
PONTOS = {
    "Lab":       (1.0,  1.0),
    "PT-TT201":  (3.5,  5.0),   # Pós-torra
    "PT-WT301":  (7.0,  1.5),   # Dosagem amendoim
    "PT-HOM301": (13.0, 4.5),   # Massa homogeneizada
    "PT-PRN401": (17.0, 4.5),   # Paçoca prensada
}

print(f"Mapa: {MAP_W} × {MAP_H} m | {len(OBSTACULOS)} obstáculos | {len(PONTOS)} pontos de interesse")

In [ ]:
def ponto_livre(x: float, y: float, obs_extra=None) -> bool:
    """Retorna True se (x, y) está dentro do mapa e fora de todos os obstáculos."""
    if not (R_AGV <= x <= MAP_W - R_AGV and R_AGV <= y <= MAP_H - R_AGV):
        return False
    todos = OBSTACULOS + (obs_extra or [])
    return not any(o.colide(x, y) for o in todos)

def segmento_livre(x1, y1, x2, y2, n_checks=20, obs_extra=None) -> bool:
    """Verifica colisão ao longo do segmento por n_checks amostras uniformes."""
    for i in range(n_checks + 1):
        t = i / n_checks
        x = x1 + t * (x2 - x1)
        y = y1 + t * (y2 - y1)
        if not ponto_livre(x, y, obs_extra):
            return False
    return True

# Verificação rápida: todos os pontos de interesse devem estar livres
for nome, (px, py) in PONTOS.items():
    status = "✓ LIVRE" if ponto_livre(px, py) else "✗ COLIDE"
    print(f"  {nome:12s} ({px:.1f}, {py:.1f}) → {status}")

---
## 3. Visualização do Mapa

In [ ]:
def plot_mapa(ax, titulo="Mapa da Fábrica de Paçoca", obs_extra=None):
    """Desenha o mapa base com obstáculos e pontos de interesse."""
    ax.set_xlim(0, MAP_W)
    ax.set_ylim(0, MAP_H)
    ax.set_aspect('equal')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')

    cores_obs = ['#c0392b', '#e67e22', '#8e44ad', '#2980b9',
                 '#7f8c8d', '#7f8c8d', '#7f8c8d', '#7f8c8d']

    todos = OBSTACULOS + (obs_extra or [])
    for i, o in enumerate(todos):
        cor = cores_obs[i] if i < len(cores_obs) else '#7f8c8d'
        if isinstance(o, RectObs):
            rect = mpatches.FancyBboxPatch(
                (o.cx - o.w/2, o.cy - o.h/2), o.w, o.h,
                boxstyle="square,pad=0", linewidth=1.5,
                edgecolor='black', facecolor=cor, alpha=0.6
            )
            ax.add_patch(rect)
            ax.text(o.cx, o.cy, o.label.split('(')[0].strip(),
                    ha='center', va='center', fontsize=7, color='white', fontweight='bold')
        elif isinstance(o, CircObs):
            circ = plt.Circle((o.cx, o.cy), o.r_obs, color=cor, alpha=0.8)
            ax.add_patch(circ)

    # Pontos de interesse
    cores_pt = {'Lab': '#27ae60', 'PT-TT201': '#e74c3c',
                'PT-WT301': '#f39c12', 'PT-HOM301': '#8e44ad', 'PT-PRN401': '#2980b9'}
    for nome, (px, py) in PONTOS.items():
        ax.plot(px, py, 'o', markersize=10, color=cores_pt.get(nome, 'k'), zorder=5)
        ax.annotate(nome, (px, py), textcoords="offset points",
                    xytext=(6, 6), fontsize=8, color=cores_pt.get(nome, 'k'), fontweight='bold')

    ax.axhline(0, color='k', lw=2)
    ax.axhline(MAP_H, color='k', lw=2)
    ax.axvline(0, color='k', lw=2)
    ax.axvline(MAP_W, color='k', lw=2)

fig, ax = plt.subplots(figsize=(13, 8))
plot_mapa(ax, "Mapa Fabril — Núcleo de Processamento (Corredores AGV)")
plt.tight_layout()
plt.show()
print("Mapa renderizado com obstáculos dos Setores 200–400 e pontos de coleta do AGV.")

---
## 4. Implementação do RRT

### 4.1 Estrutura de Dados — Nó da Árvore

Cada nó armazena:
- Configuração $(x, y)$
- Referência ao **pai** (índice no vetor de nós)
- **Custo acumulado** desde a raiz (usado pelo RRT\*)

In [ ]:
@dataclass
class No:
    x: float
    y: float
    pai: Optional[int] = None   # índice do nó pai em ArvoreRRT.nos
    custo: float = 0.0          # custo acumulado desde a raiz (RRT*)

    def dist(self, outro: 'No') -> float:
        return math.hypot(self.x - outro.x, self.y - outro.y)

    def dist_xy(self, x: float, y: float) -> float:
        return math.hypot(self.x - x, self.y - y)


class ArvoreRRT:
    """Árvore RRT / RRT* para o mapa 2D da fábrica de paçoca."""

    def __init__(self, q_init: Tuple[float, float],
                 eps: float = 0.5,
                 p_goal: float = 0.10,
                 rrt_star: bool = False,
                 r_vizinho: float = 1.5,
                 obs_extra=None,
                 seed: int = SEED):
        """
        q_init    : configuração inicial (x, y)
        eps       : passo máximo de expansão [m]
        p_goal    : probabilidade de amostrar o objetivo
        rrt_star  : ativa escolha de pai ótimo e rewiring
        r_vizinho : raio de vizinhança para RRT*
        obs_extra : lista de obstáculos adicionais (replanejamento dinâmico)
        seed      : semente para reprodutibilidade
        """
        self.eps = eps
        self.p_goal = p_goal
        self.rrt_star = rrt_star
        self.r_vizinho = r_vizinho
        self.obs_extra = obs_extra or []
        self.rng = random.Random(seed)
        self.nos: List[No] = [No(*q_init)]   # nó 0 = raiz
        self.caminho: Optional[List[int]] = None  # índices da raiz à meta

    # ------------------------------------------------------------------
    # Primitivas internas
    # ------------------------------------------------------------------
    def _mais_proximo(self, x: float, y: float) -> int:
        """Retorna o índice do nó mais próximo de (x, y)."""
        return min(range(len(self.nos)), key=lambda i: self.nos[i].dist_xy(x, y))

    def _vizinhos(self, x: float, y: float) -> List[int]:
        """Retorna índices de todos os nós a distância ≤ r_vizinho de (x, y)."""
        return [i for i, n in enumerate(self.nos) if n.dist_xy(x, y) <= self.r_vizinho]

    def _steer(self, idx_near: int, x_rand: float, y_rand: float) -> Tuple[float, float]:
        """Avança eps a partir do nó near em direção a (x_rand, y_rand)."""
        near = self.nos[idx_near]
        d = near.dist_xy(x_rand, y_rand)
        if d < 1e-9:
            return near.x, near.y
        t = min(self.eps, d) / d
        return near.x + t * (x_rand - near.x), near.y + t * (y_rand - near.y)

    def _livre(self, x1, y1, x2, y2) -> bool:
        return segmento_livre(x1, y1, x2, y2, obs_extra=self.obs_extra)

    def _livre_ponto(self, x, y) -> bool:
        return ponto_livre(x, y, self.obs_extra)

    # ------------------------------------------------------------------
    # Expansão
    # ------------------------------------------------------------------
    def _expandir(self, x_rand: float, y_rand: float) -> Optional[int]:
        """Tenta adicionar um nó novo. Retorna o índice do novo nó ou None."""
        idx_near = self._mais_proximo(x_rand, y_rand)
        near = self.nos[idx_near]
        x_new, y_new = self._steer(idx_near, x_rand, y_rand)

        if not self._livre_ponto(x_new, y_new):
            return None
        if not self._livre(near.x, near.y, x_new, y_new):
            return None

        custo_new = near.custo + near.dist_xy(x_new, y_new)

        if self.rrt_star:
            # Escolha do pai ótimo
            vizinhos = self._vizinhos(x_new, y_new)
            idx_pai = idx_near
            for idx_v in vizinhos:
                v = self.nos[idx_v]
                custo_via_v = v.custo + v.dist_xy(x_new, y_new)
                if custo_via_v < custo_new and self._livre(v.x, v.y, x_new, y_new):
                    custo_new = custo_via_v
                    idx_pai = idx_v

            novo = No(x_new, y_new, pai=idx_pai, custo=custo_new)
            idx_novo = len(self.nos)
            self.nos.append(novo)

            # Rewiring
            for idx_v in vizinhos:
                if idx_v == idx_pai:
                    continue
                v = self.nos[idx_v]
                custo_via_novo = custo_new + novo.dist_xy(v.x, v.y)
                if custo_via_novo < v.custo and self._livre(x_new, y_new, v.x, v.y):
                    self.nos[idx_v].pai = idx_novo
                    self.nos[idx_v].custo = custo_via_novo
        else:
            novo = No(x_new, y_new, pai=idx_near, custo=custo_new)
            idx_novo = len(self.nos)
            self.nos.append(novo)

        return idx_novo

    # ------------------------------------------------------------------
    # Execução principal
    # ------------------------------------------------------------------
    def planejar(self, q_goal: Tuple[float, float], n_max: int = 5000) -> Optional[List[Tuple[float, float]]]:
        """
        Executa o RRT (ou RRT*) por até n_max iterações.
        Retorna o caminho como lista de (x, y) se encontrado, ou None.
        """
        x_goal, y_goal = q_goal
        for it in range(n_max):
            # Amostragem com viés de meta
            if self.rng.random() < self.p_goal:
                x_rand, y_rand = x_goal, y_goal
            else:
                x_rand = self.rng.uniform(0, MAP_W)
                y_rand = self.rng.uniform(0, MAP_H)

            idx_novo = self._expandir(x_rand, y_rand)
            if idx_novo is None:
                continue

            novo = self.nos[idx_novo]
            if novo.dist_xy(x_goal, y_goal) <= self.eps:
                # Conecta ao objetivo
                if self._livre(novo.x, novo.y, x_goal, y_goal):
                    custo_goal = novo.custo + novo.dist_xy(x_goal, y_goal)
                    idx_goal = len(self.nos)
                    self.nos.append(No(x_goal, y_goal, pai=idx_novo, custo=custo_goal))
                    self.caminho = self._reconstruir(idx_goal)
                    if not self.rrt_star:
                        return self._nos_para_coords(self.caminho)

        # RRT* retorna o melhor caminho encontrado até N iterações
        if self.caminho is not None:
            return self._nos_para_coords(self.caminho)
        return None

    def _reconstruir(self, idx_final: int) -> List[int]:
        """Reconstrói o caminho da raiz ao nó final via ponteiros pai."""
        caminho = []
        idx = idx_final
        while idx is not None:
            caminho.append(idx)
            idx = self.nos[idx].pai
        caminho.reverse()
        return caminho

    def _nos_para_coords(self, indices: List[int]) -> List[Tuple[float, float]]:
        return [(self.nos[i].x, self.nos[i].y) for i in indices]

    def custo_caminho(self) -> float:
        """Retorna o custo total do caminho encontrado (comprimento em metros)."""
        if self.caminho is None or len(self.caminho) < 2:
            return float('inf')
        total = 0.0
        for i in range(len(self.caminho) - 1):
            a, b = self.nos[self.caminho[i]], self.nos[self.caminho[i+1]]
            total += a.dist(b)
        return total

print("Classe ArvoreRRT implementada (RRT e RRT*).")

---
## 5. Verificação das Propriedades Formais de Árvore

Após cada execução, verificamos que $T = (V, E)$ satisfaz:
1. **Conexidade:** todos os nós alcançáveis desde a raiz (BFS).
2. **Aciclia:** $|E| = |V| - 1$.
3. **Raiz única:** apenas o nó 0 tem `pai = None`.
4. **Livre de colisão:** nenhuma aresta toca $\mathcal{C}_{\text{obs}}$.

In [ ]:
def verificar_arvore(arvore: ArvoreRRT, obs_extra=None) -> Dict[str, bool]:
    """Verifica propriedades formais da árvore RRT."""
    nos = arvore.nos
    N = len(nos)

    # 1. Raiz única
    raizes = [i for i, n in enumerate(nos) if n.pai is None]
    raiz_unica = (len(raizes) == 1 and raizes[0] == 0)

    # 2. Aciclia: |E| = N - 1
    arestas = sum(1 for n in nos if n.pai is not None)
    aciclica = (arestas == N - 1)

    # 3. Conexidade: BFS a partir da raiz (via lista de filhos)
    filhos: Dict[int, List[int]] = {i: [] for i in range(N)}
    for i, n in enumerate(nos):
        if n.pai is not None:
            filhos[n.pai].append(i)
    visitados = set()
    fila = deque([0])
    while fila:
        cur = fila.popleft()
        visitados.add(cur)
        fila.extend(filhos[cur])
    conexa = (len(visitados) == N)

    # 4. Livre de colisão (amostragem de 10 pontos por aresta)
    livre = True
    for i, n in enumerate(nos):
        if n.pai is None:
            continue
        pai = nos[n.pai]
        if not segmento_livre(pai.x, pai.y, n.x, n.y, n_checks=10, obs_extra=obs_extra):
            livre = False
            break

    return {
        "Raiz única (nó 0 sem pai)": raiz_unica,
        "Acíclica (|E| = |V| - 1)": aciclica,
        "Conexa (todos alcançáveis da raiz)": conexa,
        "Livre de colisão (todas as arestas)": livre,
    }


def imprimir_verificacao(resultado: Dict[str, bool]):
    print("\n  Verificação das propriedades formais de árvore:")
    for prop, ok in resultado.items():
        simbolo = "✓" if ok else "✗"
        print(f"  {simbolo} {prop}")
    all_ok = all(resultado.values())
    print(f"\n  → {'TODAS as propriedades satisfeitas ✓' if all_ok else 'FALHA em alguma propriedade ✗'}")

print("Verificador de propriedades implementado.")

---
## 6. Execução do RRT: Lab → PT-PRN401

Primeira demonstração: trajetória do maior percurso do AGV (ponta a ponta do mapa).

In [ ]:
Q_INIT = PONTOS["Lab"]
Q_GOAL = PONTOS["PT-PRN401"]

print("=" * 60)
print(f"  RRT: {Q_INIT} → {Q_GOAL}")
print("=" * 60)

t0 = time.perf_counter()
rrt = ArvoreRRT(Q_INIT, eps=0.5, p_goal=0.10, rrt_star=False, seed=SEED)
caminho_rrt = rrt.planejar(Q_GOAL, n_max=5000)
t_rrt = time.perf_counter() - t0

if caminho_rrt:
    print(f"  Caminho encontrado em {t_rrt*1000:.1f} ms")
    print(f"  Nós na árvore: {len(rrt.nos)}")
    print(f"  Nós no caminho: {len(caminho_rrt)}")
    print(f"  Comprimento do caminho: {rrt.custo_caminho():.2f} m")
    imprimir_verificacao(verificar_arvore(rrt))
else:
    print("  Caminho NÃO encontrado em 5000 iterações.")

In [ ]:
def plot_rrt(arvore: ArvoreRRT, caminho: Optional[List[Tuple[float, float]]],
             titulo: str, obs_extra=None):
    """Visualiza a árvore RRT e o caminho encontrado sobre o mapa da fábrica."""
    fig, ax = plt.subplots(figsize=(13, 8))
    plot_mapa(ax, titulo, obs_extra=obs_extra)

    # Arestas da árvore
    for n in arvore.nos:
        if n.pai is not None:
            pai = arvore.nos[n.pai]
            ax.plot([pai.x, n.x], [pai.y, n.y], 'c-', lw=0.4, alpha=0.4)

    # Nós da árvore
    xs = [n.x for n in arvore.nos]
    ys = [n.y for n in arvore.nos]
    ax.scatter(xs, ys, s=4, c='steelblue', alpha=0.5, zorder=3)

    # Caminho encontrado
    if caminho and len(caminho) >= 2:
        cx = [p[0] for p in caminho]
        cy = [p[1] for p in caminho]
        ax.plot(cx, cy, 'r-', lw=2.5, zorder=6, label=f'Caminho ({len(caminho)} pts)')
        ax.plot(cx[0], cy[0], 'g^', ms=12, zorder=7, label='Início')
        ax.plot(cx[-1], cy[-1], 'r*', ms=14, zorder=7, label='Destino')

    ax.legend(loc='lower right', fontsize=9)
    info = f"|V|={len(arvore.nos)}  |E|={len(arvore.nos)-1}  L={arvore.custo_caminho():.1f}m"
    ax.text(0.5, 0.01, info, transform=ax.transAxes, ha='center', fontsize=9,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray'))
    plt.tight_layout()
    plt.show()

plot_rrt(rrt, caminho_rrt, "RRT — Árvore e Caminho: Lab → PT-PRN401")

---
## 7. Execução do RRT\* e Comparativo de Custo

O RRT\* continua otimizando o caminho via *rewiring* após encontrar a primeira solução.

In [ ]:
print("=" * 60)
print(f"  RRT*: {Q_INIT} → {Q_GOAL}")
print("=" * 60)

resultados = []
for n_max in [1000, 2000, 3000, 5000]:
    t0 = time.perf_counter()
    rrt_star = ArvoreRRT(Q_INIT, eps=0.5, p_goal=0.10,
                          rrt_star=True, r_vizinho=1.5, seed=SEED)
    cam = rrt_star.planejar(Q_GOAL, n_max=n_max)
    dt = time.perf_counter() - t0
    custo = rrt_star.custo_caminho() if cam else float('inf')
    resultados.append({
        "Algoritmo": f"RRT* (N={n_max})",
        "Nós |V|": len(rrt_star.nos),
        "Comprimento [m]": round(custo, 2),
        "Tempo [ms]": round(dt * 1000, 1),
        "Caminho encontrado": "Sim" if cam else "Não"
    })

# Adiciona RRT padrão para comparação
resultados.insert(0, {
    "Algoritmo": "RRT (padrão)",
    "Nós |V|": len(rrt.nos),
    "Comprimento [m]": round(rrt.custo_caminho(), 2),
    "Tempo [ms]": round(t_rrt * 1000, 1),
    "Caminho encontrado": "Sim" if caminho_rrt else "Não"
})

df_comp = pd.DataFrame(resultados)
print(df_comp.to_string(index=False))

In [ ]:
# Gráfico: custo do caminho × N para RRT*
ns = [1000, 2000, 3000, 5000]
custos_star = [r['Comprimento [m]'] for r in resultados if 'RRT*' in r['Algoritmo']]
custo_rrt_base = resultados[0]['Comprimento [m]']

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, custos_star, 'bo-', lw=2, ms=8, label='RRT* (custo decresce com N)')
ax.axhline(custo_rrt_base, color='r', linestyle='--', lw=1.5, label=f'RRT padrão ({custo_rrt_base:.1f} m)')
ax.set_xlabel('Número máximo de iterações N')
ax.set_ylabel('Comprimento do caminho [m]')
ax.set_title('Convergência do RRT* — Otimalidade Assintótica')
ax.legend()
plt.tight_layout()
plt.show()
print("\nO RRT* converge para um custo menor conforme N aumenta (otimalidade assintótica).")

---
## 8. Demonstração do Voronoi Bias

Comparamos o crescimento da árvore com $p_{\text{goal}} = 0$ (sem viés) vs. $p_{\text{goal}} = 0{,}10$ (com viés) em 500 iterações, mostrando que o viés de meta reduz o número de nós necessários.

In [ ]:
N_TESTE = 500
resultados_vies = []

for p_g, label in [(0.0, "Sem viés (p=0)"), (0.05, "Baixo viés (p=0.05)"),
                   (0.10, "Viés padrão (p=0.10)"), (0.20, "Alto viés (p=0.20)")]:
    tentativas = []
    for s in range(20):   # 20 rodadas com sementes diferentes
        a = ArvoreRRT(Q_INIT, eps=0.5, p_goal=p_g, rrt_star=False, seed=s)
        cam = a.planejar(Q_GOAL, n_max=N_TESTE)
        tentativas.append(len(a.nos) if cam else None)
    sucessos = [t for t in tentativas if t is not None]
    resultados_vies.append({
        "Configuração": label,
        "Taxa de sucesso": f"{len(sucessos)/20*100:.0f}%",
        "Média de nós (sucessos)": f"{np.mean(sucessos):.0f}" if sucessos else "—",
        "Desvio padrão": f"{np.std(sucessos):.0f}" if len(sucessos) > 1 else "—",
    })

df_vies = pd.DataFrame(resultados_vies)
print("Impacto do viés de meta (p_goal) em 20 rodadas, N_max =", N_TESTE)
print(df_vies.to_string(index=False))
print("\nConclusão: p=0.10 equilíbrio entre exploração global (Voronoi) e direção à meta.")

---
## 9. Replanejamento Dinâmico: Porta XV-PORT-200 Fecha Durante a Rota

Simulamos o cenário operacional da Aula 22 (seção 6.1 do `.md`):
o AGV saiu do `Lab` em direção a `PT-TT201`. A meio caminho, o SCADA detecta
que `XV-PORT-200 = 0` (porta do Setor 200 fechada durante ciclo de torra).
O módulo RRT recalcula a rota a partir da posição atual.

In [ ]:
# Obstáculo dinâmico: porta fechada (corredor entre S100 e S200)
# Modela um bloqueio retangular estreito na passagem
PORTA_FECHADA = RectObs(cx=2.5, cy=5.5, w=0.5, h=4.0, label="XV-PORT-200 (FECHADA)")

# Posição do AGV quando o SCADA dispara o replanejamento
# (a meio caminho Lab → PT-TT201)
POS_ATUAL_AGV = (1.8, 3.0)

print("=" * 60)
print("  REPLANEJAMENTO DINÂMICO")
print(f"  Posição atual do AGV: {POS_ATUAL_AGV}")
print(f"  Destino: PT-TT201 {PONTOS['PT-TT201']}")
print(f"  Novo obstáculo: XV-PORT-200 fechada em x=2.5, y∈[3.5,7.5]")
print("=" * 60)

# Rota original (sem obstáculo dinâmico)
rrt_orig = ArvoreRRT(Q_INIT, eps=0.5, p_goal=0.10, seed=SEED)
cam_orig = rrt_orig.planejar(PONTOS["PT-TT201"], n_max=5000)

# Replanejamento a partir da posição atual (com porta fechada)
t0 = time.perf_counter()
rrt_rep = ArvoreRRT(POS_ATUAL_AGV, eps=0.5, p_goal=0.10, seed=SEED,
                     obs_extra=[PORTA_FECHADA])
cam_rep = rrt_rep.planejar(PONTOS["PT-TT201"], n_max=5000)
t_rep = time.perf_counter() - t0

if cam_rep:
    print(f"  Replanejamento: {t_rep*1000:.1f} ms | {len(rrt_rep.nos)} nós | {rrt_rep.custo_caminho():.2f} m")
    print("  AGV encontrará caminho alternativo contornando a porta fechada.")
    imprimir_verificacao(verificar_arvore(rrt_rep, obs_extra=[PORTA_FECHADA]))
else:
    print("  ALARME: Nenhum caminho encontrado. AGV permanece em posição segura (ALM-AGV=1).")

In [ ]:
# Visualização comparativa: rota original vs. replanejada
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

plot_mapa(axes[0], "Rota original: Lab → PT-TT201")
if cam_orig:
    cx = [p[0] for p in cam_orig]; cy = [p[1] for p in cam_orig]
    for n in rrt_orig.nos:
        if n.pai is not None:
            p = rrt_orig.nos[n.pai]
            axes[0].plot([p.x, n.x], [p.y, n.y], 'c-', lw=0.4, alpha=0.3)
    axes[0].plot(cx, cy, 'g-', lw=2.5, label='Rota original')
    axes[0].plot(cx[0], cy[0], 'g^', ms=12); axes[0].plot(cx[-1], cy[-1], 'r*', ms=14)
axes[0].legend(fontsize=9)

plot_mapa(axes[1], "Replanejamento: posição atual → PT-TT201 (porta fechada)",
          obs_extra=[PORTA_FECHADA])
axes[1].plot(*POS_ATUAL_AGV, 'ys', ms=14, zorder=7, label='Posição atual AGV')
if cam_rep:
    cx2 = [p[0] for p in cam_rep]; cy2 = [p[1] for p in cam_rep]
    for n in rrt_rep.nos:
        if n.pai is not None:
            p = rrt_rep.nos[n.pai]
            axes[1].plot([p.x, n.x], [p.y, n.y], 'c-', lw=0.4, alpha=0.3)
    axes[1].plot(cx2, cy2, 'm-', lw=2.5, label='Nova rota (replanejada)')
    axes[1].plot(cx2[-1], cy2[-1], 'r*', ms=14)
axes[1].legend(fontsize=9)

plt.suptitle("Replanejamento dinâmico — XV-PORT-200 fecha durante operação",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Benchmarking: Rota Completa de Coleta (Sequência da Aula 17)

Usando a sequência ótima do TSP (Aula 17): `Lab → PT-WT301 → PT-TT201 → PT-HOM301 → PT-PRN401 → Lab`,
planejamos cada segmento com RRT e calculamos o comprimento total real (evitando obstáculos).

In [ ]:
# Sequência de visitas (resultado TSP da Aula 17 adaptado ao mapa 2D)
SEQUENCIA_TSP = [
    "Lab", "PT-WT301", "PT-TT201", "PT-HOM301", "PT-PRN401", "Lab"
]

print("=" * 60)
print("  ROTA COMPLETA DE COLETA DE AMOSTRAS (AGV)")
print("  Sequência:", " → ".join(SEQUENCIA_TSP))
print("=" * 60)

seg_data = []
arvores_rota = []
caminhos_rota = []
custo_total = 0.0

for i in range(len(SEQUENCIA_TSP) - 1):
    orig = PONTOS[SEQUENCIA_TSP[i]]
    dest = PONTOS[SEQUENCIA_TSP[i + 1]]
    dist_eucl = math.hypot(dest[0] - orig[0], dest[1] - orig[1])

    t0 = time.perf_counter()
    arv = ArvoreRRT(orig, eps=0.5, p_goal=0.10, seed=SEED + i)
    cam = arv.planejar(dest, n_max=5000)
    dt = time.perf_counter() - t0

    custo_seg = arv.custo_caminho() if cam else float('inf')
    custo_total += custo_seg if custo_seg != float('inf') else 0

    arvores_rota.append(arv)
    caminhos_rota.append(cam)
    seg_data.append({
        "Segmento": f"{SEQUENCIA_TSP[i]} → {SEQUENCIA_TSP[i+1]}",
        "Dist. Euclidiana [m]": round(dist_eucl, 2),
        "Comprimento RRT [m]": round(custo_seg, 2) if custo_seg != float('inf') else "—",
        "Fator desvio": round(custo_seg / dist_eucl, 2) if custo_seg != float('inf') else "—",
        "Nós |V|": len(arv.nos),
        "Tempo [ms]": round(dt * 1000, 1),
    })

df_rota = pd.DataFrame(seg_data)
print(df_rota.to_string(index=False))
print(f"\n  Comprimento total da rota RRT: {custo_total:.2f} m")
print(f"  (Distância euclidiana total sem obstáculos: {sum(r['Dist. Euclidiana [m]'] for r in seg_data):.2f} m)")

In [ ]:
# Visualização da rota completa de coleta
fig, ax = plt.subplots(figsize=(13, 8))
plot_mapa(ax, "Rota Completa de Coleta do AGV — RRT por Segmento")

cores_rota = ['#e74c3c', '#f39c12', '#27ae60', '#8e44ad', '#2980b9']
for i, (cam, cor) in enumerate(zip(caminhos_rota, cores_rota)):
    if cam and len(cam) >= 2:
        cx = [p[0] for p in cam]
        cy = [p[1] for p in cam]
        label = f"{SEQUENCIA_TSP[i]} → {SEQUENCIA_TSP[i+1]}"
        ax.plot(cx, cy, '-', lw=2.5, color=cor, label=label, zorder=5)

ax.legend(loc='lower right', fontsize=8)
ax.set_title(f"Rota completa AGV | Comprimento total RRT: {custo_total:.1f} m", fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Síntese: RRT como Árvore em Matemática Discreta

Consolidamos as propriedades formais verificadas ao longo do notebook.

In [ ]:
print("=" * 70)
print("  SÍNTESE FORMAL — RRT como Árvore (Matemática Discreta)")
print("=" * 70)

sintese = [
    ("T = (V, E) acíclico e conexo", "Verificado em todas as execuções (|E| = |V|-1)"),
    ("Raiz q_init", "Único nó sem pai; ponto de acesso à árvore"),
    ("Nó folha em q_goal", "Vértice de grau 1 terminal quando caminho encontrado"),
    ("Voronoi Bias", "Nós de fronteira têm maior prob. de expansão → exploração rápida"),
    ("Rewiring (RRT*)", "Análogo às rotações AVL (Aula 21): reequilibra custo, não altura"),
    ("Completude probabilística", "P(solução) → 1 quando N → ∞ (provado matematicamente)"),
    ("Livre de colisão", "Garantido pela verificação antes de cada inserção de aresta"),
]

print(f"\n{'Propriedade formal':<35} {'Verificação / Significado'}")
print("-" * 70)
for prop, verif in sintese:
    print(f"  {prop:<33} {verif}")

print("\n" + "=" * 70)
print("  RESUMO DOS RESULTADOS")
print("=" * 70)
print(f"  RRT  padrão  Lab→PT-PRN401: {rrt.custo_caminho():.2f} m | {len(rrt.nos)} nós")

# RRT* final para o resumo
rrt_star_final = ArvoreRRT(Q_INIT, eps=0.5, p_goal=0.10, rrt_star=True, r_vizinho=1.5, seed=SEED)
cam_star_final = rrt_star_final.planejar(Q_GOAL, n_max=5000)
print(f"  RRT* (N=5000) Lab→PT-PRN401: {rrt_star_final.custo_caminho():.2f} m | {len(rrt_star_final.nos)} nós")
print(f"  Rota completa de coleta (5 segmentos): {custo_total:.2f} m")
print(f"  Replanejamento dinâmico (XV-PORT-200): {'Sim' if cam_rep else 'Não'} | {t_rep*1000:.1f} ms")
print("\n  Todas as propriedades de árvore verificadas: ✓")

---
## 12. Exemplo Resolvido (reprodução analítica do .md)

**Verificação numérica do Steering e detecção de colisão do nó 158 → 159.**

In [ ]:
print("=" * 60)
print("  EXEMPLO RESOLVIDO — Verificação do Exemplo do .md")
print("=" * 60)

q_near = (11.3, 5.8)
q_rand = (14.0, 7.0)
eps_ex = 0.5

d = math.hypot(q_rand[0] - q_near[0], q_rand[1] - q_near[1])
t = eps_ex / d
q_new = (q_near[0] + t * (q_rand[0] - q_near[0]),
          q_near[1] + t * (q_rand[1] - q_near[1]))

print(f"  q_near = {q_near}")
print(f"  q_rand = {q_rand}")
print(f"  dist(q_near, q_rand) = {d:.4f} m")
print(f"  q_new  = ({q_new[0]:.3f}, {q_new[1]:.3f})")

# Verificação de colisão com HOM-301 considerando r_AGV
obs_hom = RectObs(13.0, 7.0, 2.5, 2.5, "HOM-301")
colide = obs_hom.colide(*q_new, r=R_AGV)
print(f"\n  Colisão com HOM-301 (r_AGV={R_AGV} m): {'✗ COLIDE → aresta REJEITADA' if colide else '✓ Livre'}")

# Fronteira expandida
x_front = obs_hom.cx - obs_hom.w / 2 - R_AGV
print(f"  Fronteira efetiva do HOM-301 (com r_AGV): x ≥ {obs_hom.cx - obs_hom.w/2:.2f} m → expandida para x ≥ {x_front:.2f} m")
print(f"  q_new.x = {q_new[0]:.3f} > {x_front:.3f}? {'Sim → dentro da zona proibida' if q_new[0] > x_front else 'Não → livre'}")

---
## 13. Entregável

Este notebook implementou e validou:

| Item | Resultado |
| :--- | :--- |
| **Mapa da fábrica** | 8 obstáculos (4 retangulares + 4 circulares), 5 pontos de interesse |
| **Verificador de colisão** | `ponto_livre` e `segmento_livre` com margem $r_{AGV}$ |
| **Classe `ArvoreRRT`** | RRT e RRT\* (pai ótimo + rewiring) em Python puro |
| **Verificador de propriedades** | Raiz única, aciclia $|E|=|V|-1$, conexidade (BFS), livre de colisão |
| **Comparativo RRT vs. RRT\*** | Custo decresce com $N$ (otimalidade assintótica confirmada) |
| **Rota completa de coleta** | 5 segmentos (sequência TSP da Aula 17) planejados com RRT |
| **Replanejamento dinâmico** | XV-PORT-200 fecha → novo caminho calculado em < 130 ms |
| **Exemplo resolvido** | Steering e detecção de colisão com HOM-301 verificados analiticamente |

> **Conexão com as demais aulas da Etapa 03:**  
> A árvore RRT compartilha a estrutura formal $(V, E)$ acíclica e conexa com a Árvore de Huffman (Aula 20), a BDT/AVL (Aula 21) e a Árvore Geradora do RSTP (Aula 23).  
> O RRT\* realiza *rewiring* (reconfiguração de arestas pai) analogamente às **rotações AVL** da Aula 21.  
> O planejamento de trajetória **complementa** o roteamento hamiltoniano (TSP, Aula 17) adicionando a camada de navegação livre de colisões no espaço contínuo.